# Writing actions as code snippets or JSON blobs

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies and login to our HF account to access the Inference API

If you haven't installed `smolagents` yet, you can do so by running the following command:

In [ ]:
!pip install smolagents -U

Let's also login to the Hugging Face Hub to have access to the Inference API.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from huggingface_hub import HfApi

try:
    user_info = HfApi().whoami()
    print(f"OK! Logged in as: {user_info}")
except:
    print(f"ERROR!")

## Selecting a Playlist for the Party Using `smolagents` and a `ToolCallingAgent`

Let's revisit the previous example where Alfred started party preparations, but this time we'll use a `ToolCallingAgent` to highlight the difference. We'll build an agent that can search the web, just like in our Code Agent example. The only difference is the agent type - the framework handles everything else:

In [ ]:
# Setup Langfuse tracing
import os
import base64
import yaml

with open("../../../api_keys/langfuse.yaml", "r") as stream:
    data = yaml.safe_load(stream)
    LANGFUSE_PUBLIC_KEY = data["public"]
    LANGFUSE_SECRET_KEY = data["secret"]
LANGFUSE_AUTH=base64.b64encode(f"{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}".encode()).decode()

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://cloud.langfuse.com/api/public/otel" # EU data region
# os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://us.cloud.langfuse.com/api/public/otel" # US data region
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"

In [ ]:
from opentelemetry.sdk.trace import TracerProvider

from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

trace_provider = TracerProvider()
trace_provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter()))

SmolagentsInstrumentor().instrument(tracer_provider=trace_provider)

In [ ]:
from smolagents import ToolCallingAgent, WebSearchTool, InferenceClientModel

# ERROR: need to find a provider which supports a ToolCallingAgent

agent = ToolCallingAgent(
    tools=[WebSearchTool()],
    model=InferenceClientModel(
        model_id="Qwen/Qwen2.5-32B-Instruct",
        provider="featherless-ai",
    )
)

agent.run("Search for the best music recommendations for a party at the Wayne's mansion.")

Observations:
- Qwen2.5-7B fails to produce a correct tool call at step N.
- the framework correct for this by sending an ERROR message, and prompting the model to retry.
- the model then succesfully completes the task.

In [ ]:
import pprint
pprint.pp(agent.prompt_templates)


When you examine the agent's trace, instead of seeing `Executing parsed code:`, you'll see something like:

```text
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'web_search' with arguments: {'query': "best music recommendations for a party at Wayne's         │
│ mansion"}                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
```  

The agent generates a structured tool call that the system processes to produce the output, rather than directly executing code like a `CodeAgent`.

Now that we understand both agent types, we can choose the right one for our needs. Let's continue exploring `smolagents` to make Alfred's party a success! 🎉